# AValidate · Audio inference bundle for the app

Packages the final audio model (`beats_lora_r8_v1`) into one folder the app can load (`MyDrive/AValidate/inference/audio/`), then proves that **live predictions equal the offline export** (`embeddings/audio_final/`) on the test clips. That equality is what lets a fusion model trained on the exports run on live audio. Any GPU works; this only runs inference.

In [ ]:
!nvidia-smi -L || echo 'No GPU: switch the runtime to GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
from google.colab import userdata

BRANCH = 'audio/inference'   # switch to 'main' once this branch is merged
REPO = '/content/Audio-Video-Confidence'
token = userdata.get('GH_TOKEN')
url = f'https://{token}@github.com/NUS-AIS-GCs/Audio-Video-Confidence.git'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO], check=True)
else:
    # fetch + hard reset (not pull): always matches GitHub exactly, even if branch history was rewritten
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '-q', '--hard', f'origin/{BRANCH}'], check=True)
os.chdir(REPO)
!git log --oneline -1

In [ ]:
!pip -q install -r requirements-audio.txt

## 1 · Dataset (videos are needed to run the model on real clips)

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.data_ave

## 2 · Build the inference bundle
Copies the LoRA + head weights, the BEATs checkpoint, normalisation stats and class names, with a model card and checksums (about 360 MB).

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.export_inference --run-id beats_lora_r8_v1

## 3 · Consistency check: live inference vs the exported `audio_final` rows (all 402 test clips)
Expect max |Δprob| around 1e-3 or less (float16 storage rounding in the export) and identical per-second labels for (nearly) every clip.

In [ ]:
%cd /content/Audio-Video-Confidence
import numpy as np, pandas as pd
from audio.infer import AudioPredictor
from audio import config as C
p = AudioPredictor(C.DRIVE_ROOT / 'inference' / 'audio')
ex = C.EMBED_ROOT / 'audio_final'
ids = pd.read_csv(ex / 'test_ids.csv'); P = np.load(ex / 'test_probs.npy').astype(np.float32); E = np.load(ex / 'test_emb.npy').astype(np.float32)
videos = {v.stem: v for v in C.AVE_ROOT.rglob('*.mp4')}
dp = de = 0.0; same = 0
for i, c in enumerate(ids.clip_id):
    out = p.predict_video(videos[c])
    dp = max(dp, float(np.abs(out.probs - P[i]).max())); de = max(de, float(np.abs(out.embeddings - E[i]).max()))
    same += int((out.probs.argmax(1) == P[i].argmax(1)).all())
print(f'test clips: {len(ids)} | max |Δprob| {dp:.4f} | max |Δemb| {de:.4f} | identical per-second labels: {same}/{len(ids)}')

## 4 · The app's audio branch on a real clip, and the real swap test

In [ ]:
%cd /content/Audio-Video-Confidence
import os
ann = next(C.AVE_ROOT.rglob('Annotations.txt')).parent
if not os.path.exists('AVE_Dataset'): os.symlink(ann, 'AVE_Dataset')     # the app expects <repo>/AVE_Dataset
os.environ['AVAL_AUDIO_BUNDLE'] = str(C.DRIVE_ROOT / 'inference' / 'audio')
from src.models.audio import AudioBranch
from src.config import VIDEOS_DIR
b = AudioBranch()
clip = sorted(VIDEOS_DIR.glob('*.mp4'))[0]
info = b.ave_index.get_info(clip.stem)
own = b.extract_and_classify(str(clip))
print('clip', clip.stem, '| truth:', info and (info['category'], info['start_time'], info['end_time']))
print('own audio  ->', own['audio_class'], round(own['audio_confidence'], 3), own['presence'])
sw = b.extract_and_classify(str(clip), swap_audio=True)
donor = b.ave_index.get_info(sw['swap_source'])
print('swapped in ', sw['swap_source'], (donor['category'], donor['start_time'], donor['end_time']), '->', sw['audio_class'], sw['presence'])

## 5 · Using it in the app locally
Download `MyDrive/AValidate/inference/audio/` and place it at `<repo>/_models/audio/` (git-ignored), or set `AVAL_AUDIO_BUNDLE` to its path. The audio branch then runs the real model; without it, it falls back to the placeholder and warns.